# my-coding-agent — Sample Notebook

Demonstrates the  library: tool registration, single-shot LLM calls, and the full agentic loop.

**Prerequisites:** local OMLX server running at  and the package installed:


## 1. Imports

In [1]:
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

## 2. Inspect a tool definition

The  converter turns any typed Python function into an OpenAI-compatible schema.

In [2]:
import json

# Built-in example tool
schema = tool(ToolsRegistry.get_weather)
print(json.dumps(schema, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string"
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


## 3. Register a custom tool

Define a new function and register it on  at runtime.

In [3]:
def get_stock_price(ticker: str, currency: str = "USD") -> str:
    """Get the current stock price for a ticker symbol."""
    # stub — replace with a real API call
    prices = {"AAPL": 213.5, "GOOGL": 175.2, "MSFT": 425.0}
    price = prices.get(ticker.upper(), 0.0)
    return f"{ticker.upper()} is trading at {price} {currency}"

# Attach to registry so the LLM can call it
ToolsRegistry.get_stock_price = staticmethod(get_stock_price)

print(json.dumps(tool(get_stock_price), indent=2))

{
  "type": "function",
  "function": {
    "name": "get_stock_price",
    "description": "Get the current stock price for a ticker symbol.",
    "parameters": {
      "type": "object",
      "properties": {
        "ticker": {
          "type": "string"
        },
        "currency": {
          "type": "string"
        }
      },
      "required": [
        "ticker"
      ]
    }
  }
}


## 4. Single-shot LLM call (no tools)

Use  directly for a plain chat completion.

In [4]:
llm = LLM(api_url=OMLX_API_URL, api_key=OMLX_API_KEY, model=OMLX_MODEL)

messages = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user",   "content": "What is 42 in binary?"}
]

resp = llm.chat_completion(messages)
msg = extract_message(resp)
print(msg["content"])

2026-05-22 17:11:27 | INFO     | LLM | Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']
2026-05-22 17:11:27 | INFO     | LLM | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 17:11:32 | INFO     | LLM | Received response: 200 (1526 bytes)
2026-05-22 17:11:32 | DEBUG    | LLM | Response content: {'id': 'chatcmpl-f4728b20', 'object': 'chat.completion', 'created': 1779462692, 'model': 'Qwen3.6-35B-A3B-4bit', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': '42 in binary is **101010**.', 'reasoning_content': 'Here\'s a thinking process:\n\n1.  

## 5. Single-shot call with tool use

Pass tools manually and inspect the tool-call response, then execute it.

In [5]:
tools = [
    tool(ToolsRegistry.get_weather),
    tool(get_stock_price),
]

messages = [
    {"role": "system", "content": "Use the available tools to answer."},
    {"role": "user",   "content": "What is the weather in Tokyo?"}
]

resp = llm.chat_completion(messages, tools=tools)
msg = extract_message(resp)
print("finish_reason:", extract_finish_reason(resp))
print("tool_calls:", json.dumps(msg.get("tool_calls"), indent=2))

2026-05-22 17:11:32 | INFO     | LLM | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 17:11:37 | INFO     | LLM | Received response: 200 (2187 bytes)
2026-05-22 17:11:37 | DEBUG    | LLM | Response content: {'id': 'chatcmpl-89134a14', 'object': 'chat.completion', 'created': 1779462697, 'model': 'Qwen3.6-35B-A3B-4bit', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': None, 'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to know the current weather in Tokyo.\n2.  **Check Available Tools**: I have a `get_weather` tool that takes a `location` parameter.\n3.  **Map Input to Tool**: Location = "Tokyo".\n4.  **Execute Tool Call**: Call `get_weather(location="Tokyo")`.\n5.  **Formulate Response**: Wait for tool output, then answer the user. (Self-correction: I will just make the tool call now). \n   *Tool Call*: `get_weather(location="Tokyo")`\n   *Proceeds*. \n   *Output Generation*

In [6]:
# Execute the tool calls and get the final answer
messages.append(msg)
tool_results = llm.execute_tool_calls(msg)
messages.extend(tool_results)

final_resp = llm.chat_completion(messages)
print(extract_message(final_resp)["content"])

2026-05-22 17:11:37 | INFO     | LLM | Processing tool call: get_weather with args {'location': 'Tokyo'}
2026-05-22 17:11:37 | INFO     | LLM | Executed tool: get_weather with args {'location': 'Tokyo'}
2026-05-22 17:11:37 | INFO     | LLM | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 17:11:39 | INFO     | LLM | Received response: 200 (838 bytes)
2026-05-22 17:11:39 | DEBUG    | LLM | Response content: {'id': 'chatcmpl-cec872bf', 'object': 'chat.completion', 'created': 1779462699, 'model': 'Qwen3.6-35B-A3B-4bit', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'The current weather in Tokyo is sunny with a temperature of 25 degrees Celsius.', 'reasoning_content': 'The tool returned the current weather in Tokyo: sunny with a temperature of 25 degrees Celsius. I will now convey this information directly to the user.', 'tool_calls': None}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 471, 'completion_tok

## 6. Full agentic loop with 

 handles the chat → tool-execute → respond cycle automatically.

In [7]:
from my_coding_agent import LLM, Agent, tool, ToolsRegistry
from my_coding_agent.llm import OMLX_API_URL, OMLX_API_KEY, OMLX_MODEL
from my_coding_agent.utils import extract_message, extract_finish_reason

messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant. Use tools when needed. "
            "Available tools: get_weather(location), get_stock_price(ticker, currency)."
        )
    },
    {"role": "user", "content": "What is the weather in London and the price of AAPL stock?"}
]

agent = Agent(
    api_url=OMLX_API_URL,
    api_key=OMLX_API_KEY,
    model=OMLX_MODEL,
    messages=messages,
    tools=tools,
)

final_messages = agent.run(max_steps=5)
print(" Final messages: ", json.dumps(final_messages, indent=2))

2026-05-22 17:11:39 | INFO     | LLM | Available models: ['MLX-Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'MLX-Qwen3.5-9B-Claude-4.6-Opus-Reasoning-Distilled-v2-4bit', 'Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit', 'Qwen3.5-9B-MLX-4bit', 'Qwen3.6-27B-4bit', 'Qwen3.6-35B-A3B-4bit', 'Qwen3.6-35B-A3B-5bit', 'Qwen3.6-35B-A3B-6bit', 'gemma-4-26b-a4b-it-4bit', 'gemma-4-31b-it-4bit', 'gemma-4-e2b-it-4bit', 'gpt-oss-20b-MXFP4-Q4', 'gpt-oss-20b-MXFP4-Q8']
2026-05-22 17:11:39 | INFO     | LLM | Agent initialized with API URL: http://127.0.0.1:8321/v1, Model: Qwen3.6-35B-A3B-4bit
2026-05-22 17:11:39 | INFO     | LLM | Agent run started with max_steps: 5
2026-05-22 17:11:39 | INFO     | LLM | Running Agent step 1/5
2026-05-22 17:11:39 | INFO     | LLM | Request sent to http://127.0.0.1:8321/v1/chat/completions with model Qwen3.6-35B-A3B-4bit
2026-05-22 17:11:42 | INFO     | LLM | Received response: 200 (1318 bytes)
2026-05-22 17:11:42 | DEBUG    | LLM | Response content: {'id': '